<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/Question_Generation_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Train

In [ ]:
!pip install transformers datasets evaluate accelerate rouge_score sacrebleu bert_score -q

In [ ]:
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)


import evaluate

In [ ]:
dataset = load_dataset("rajpurkar/squad")
dataset

In [ ]:
dataset["train"][0]

In [ ]:
def prepare_qg_example(example):
    context = example["context"]
    question = example["question"]
    answer = example["answers"]["text"][0]

    input_text = (
        f"Generate a question based on the context.\n"
        f"Answer: {answer}\n"
        f"Context: {context}\n"
        f"Question:"
    )

    target_text = question

    return {
        "input_text": input_text,
        "target_text": target_text,
        "answer": answer,
        "context": context,
        "question": question
    }

In [ ]:
qg_dataset = dataset.map(prepare_qg_example)

In [ ]:
qg_dataset["train"][0]["input_text"]

In [ ]:
qg_dataset["train"][0]["target_text"]

In [ ]:
train_data = qg_dataset["train"].shuffle(seed=42).select(range(10000))
val_data = qg_dataset["validation"].shuffle(seed=42).select(range(1000))

In [ ]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
max_input_length = 512
max_target_length = 64

def tokenize_function(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=max_input_length,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=max_target_length,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
tokenized_train = train_data.map(
    tokenize_function,
    batched=True,
    remove_columns=train_data.column_names
)

tokenized_val = val_data.map(
    tokenize_function,
    batched=True,
    remove_columns=val_data.column_names
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")


In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    if preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    labels = np.where(labels < 0, tokenizer.pad_token_id, labels)

    preds = preds.astype(np.int64)
    labels = labels.astype(np.int64)

    decoded_preds = tokenizer.batch_decode(
        preds,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    bleu = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[label] for label in decoded_labels]
    )

    rouge = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        "bleu": bleu["score"],
        "rouge1": rouge["rouge1"],
        "rouge2": rouge["rouge2"],
        "rougeL": rouge["rougeL"],
    }

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_question_generation",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    num_train_epochs=3,
    weight_decay=0.01,

    predict_with_generate=True,
    generation_max_length=64,

    logging_steps=50,
    save_total_limit=2,

    fp16=False,

    generation_num_beams=4,


    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    report_to="none"
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ]
)

In [ ]:
trainer.train()

#Eval

In [ ]:
bertscore_metric = evaluate.load("bertscore")

In [ ]:
eval_results = trainer.evaluate()
eval_results

In [ ]:
def generate_question(context, answer):
    model.eval()

    input_text = (
        f"Generate a question based on the context.\n"
        f"Answer: {answer}\n"
        f"Context: {context}\n"
        f"Question:"
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    question = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return question

In [ ]:
sample = qg_dataset["validation"][5]

context = sample["context"]
answer = sample["answer"]
reference_question = sample["question"]

generated_question = generate_question(context, answer)


print("Context:", context)
print("ANSWER:", answer)
print("REFERENCE:", reference_question)
print("GENERATED:", generated_question)

In [ ]:
def compute_single_metrics(prediction, reference):
    bleu = bleu_metric.compute(
        predictions=[prediction],
        references=[[reference]]
    )

    rouge = rouge_metric.compute(
        predictions=[prediction],
        references=[reference]
    )

    return {
        "bleu": bleu["score"],
        "rouge1": rouge["rouge1"],
        "rouge2": rouge["rouge2"],
        "rougeL": rouge["rougeL"]
    }

In [ ]:
rows = []

predictions = []
references = []

for i in range(20):
    sample = qg_dataset["validation"][i]

    context = sample["context"]
    answer = sample["answer"]
    reference = sample["question"]

    prediction = generate_question(context, answer)
    scores = compute_single_metrics(prediction, reference)

    predictions.append(prediction)
    references.append(reference)

    rows.append({
        "context": context[:300] + "...",
        "answer": answer,
        "reference_question": reference,
        "generated_question": prediction,
        "bleu": scores["bleu"],
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"],
    })

bert_scores = bertscore_metric.compute(
    predictions=predictions,
    references=references,
    model_type="distilbert-base-uncased"
)

for i in range(len(rows)):
    rows[i]["bertscore_precision"] = bert_scores["precision"][i]
    rows[i]["bertscore_recall"] = bert_scores["recall"][i]
    rows[i]["bertscore_f1"] = bert_scores["f1"][i]

eval_df = pd.DataFrame(rows)
eval_df

#Manual

In [ ]:
eval_df["error_type"] = ""
eval_df["comment"] = ""
eval_df

eval_df.loc[0, "error_type"] = "good"
eval_df.loc[0, "comment"] = "Generated question matches the reference meaning."

eval_df.loc[1, "error_type"] = "wrong_answer_focus"
eval_df.loc[1, "comment"] = "Question is valid, but it asks about another answer."

eval_df.loc[2, "error_type"] = "low_overlap_but_good"
eval_df.loc[2, "comment"] = "Meaning is close, but wording differs, so BLEU/ROUGE are lower."

In [ ]:
eval_df[["bleu", "rouge1", "rouge2", "rougeL"]].describe()

In [ ]:
eval_df["error_type"].value_counts()

In [ ]:
eval_df.sort_values("bleu").head(5)[
    ["answer", "reference_question", "generated_question", "bleu", "rouge1", "rouge2", "rougeL", "error_type"]
]

In [ ]:
eval_df.sort_values("rougeL", ascending=False).head(5)[
    ["answer", "reference_question", "generated_question", "bleu", "rouge1", "rouge2", "rougeL", "error_type"]
]